# Autonomic multi LLM system coordinierer

Ziel ist ein autonomes System, das drei kleine Sprachmodelle parallel laufen lässt. Ihre Ergebnisse werden bewertet, um die präziseste Aussage zu priorisieren und mit wertvollen Informationen der anderen Modelle anzureichern.

In [ ]:
#!pip install sentence-transformers sympy spacy flashrank
#!python -m spacy download de_core_news_lg

In [34]:
# IMPORT
import asyncio
import os
import re
import socket
import subprocess
import time
import urllib.error
import urllib.request
import warnings
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, List, Set

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import nltk
import numpy as np
import ollama
import pandas as pd
import psutil
import spacy
from sentence_transformers import CrossEncoder
from sympy import sympify

#  Environment & Warnings 
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
from nltk.corpus import stopwords

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

try:
    nlp_de = spacy.load("de_core_news_lg")
except OSError:
    subprocess.run(["python", "-m", "spacy", "download", "de_core_news_lg"], check=True)
    nlp_de = spacy.load("de_core_news_lg")

nli_model_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
try:
    nli_model = CrossEncoder(nli_model_name, local_files_only=True)
except Exception:
    nli_model = CrossEncoder(nli_model_name, local_files_only=False)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
# Globals
_cfg: Dict[str, Any] = globals().setdefault("_AGENT_CONFIG", {})

# Netzwerk- und Ausführungs-Einstellungen für Ollama
OLLAMA_HOST: str        = _cfg.setdefault("OLLAMA_HOST", "http://127.0.0.1:11434")
SOCKET_TIMEOUT: float   = _cfg.setdefault("SOCKET_TIMEOUT", 1.0)
API_TIMEOUT: float      = _cfg.setdefault("API_TIMEOUT", 1.5)
MAX_START_RETRIES: int  = _cfg.setdefault("MAX_START_RETRIES", 5)
RETRY_WAIT_SEC: float   = _cfg.setdefault("RETRY_WAIT_SEC", 1.5)
PORT_DEFAULT: int       = _cfg.setdefault("PORT_DEFAULT", 11434)
PROJECT_MODELS_DIR: str = _cfg.setdefault("PROJECT_MODELS_DIR", "../data/models")

OLAMA_MODEL_CODE_LOW    = _cfg.setdefault("OLAMA_MODEL_CODE_LOW",    "codestral:22b")
OLAMA_MODEL_CODE_HIGH   = _cfg.setdefault("OLAMA_MODEL_CODE_HIGH",   "qwen3-coder-next-ud-q4-k-xl")
OLAMA_MODEL_LOGIC_LOW   = _cfg.setdefault("OLAMA_MODEL_LOGIC_LOW",   "deepseek-r1:8b")
OLAMA_MODEL_LOGIC_HIGH  = _cfg.setdefault("OLAMA_MODEL_LOGIC_HIGH",  "qwen3-coder-next-ud-q4-k-xl")
OLAMA_MODEL_TEXT_LOW    = _cfg.setdefault("OLAMA_MODEL_TEXT_LOW",    "gemma3:4b")
OLAMA_MODEL_TEXT_HIGH   = _cfg.setdefault("OLAMA_MODEL_TEXT_HIGH",   "qwen3.8:27b")
OLAMA_MODEL_FUSION      = _cfg.setdefault("OLAMA_MODEL_FUSION",      "qwen3.5-2b-artemis-q5-k-m")

In [36]:
# def for run
def ensure_ollama_running(ollama_host: str = None) -> bool:
    """
    Prüft autonom, ob Ollama läuft, verhindert Mehrfachstarts (und CMD-Fenster) 
    und greift sauber auf die globalen Einstellungen zu.
    """
    def is_port_open(ip, port, timeout):
        try:
            with socket.create_connection((ip, port), timeout=timeout):
                return True
        except (socket.timeout, ConnectionRefusedError, OSError):
            return False

    def is_ollama_process_running():
        """Prüft, ob bereits ein Ollama-Prozess im System läuft (verhindert neue CMD-Fenster)."""
        if os.name != "nt":
            return False
        try:
            output = subprocess.check_output("tasklist /fi \"imagename eq ollama.exe\"", shell=True, text=True)
            return "ollama.exe" in output.lower()
        except Exception:
            return False
    target_host = ollama_host or OLLAMA_HOST
    host_ip = target_host.replace("http://", "").replace("https://", "").split(":")[0]
    try:
        port = int(target_host.split(":")[-1])
    except ValueError:
        port = PORT_DEFAULT
    # 1. Schritt: Prüfen, ob der Port offen ist und die API antwortet
    if is_port_open(host_ip, port, timeout=SOCKET_TIMEOUT):
        try:
            req = urllib.request.Request(f"{target_host}/api/tags")
            with urllib.request.urlopen(req, timeout=API_TIMEOUT) as resp:
                if resp.status == 200:
                    return True
        except Exception:
            pass
    # 2. Schritt: Prozess-Check – Läuft ollama.exe bereits im System?
    if is_ollama_process_running():
        # Prozess existiert, aber Port antwortet noch nicht ganz -> Geduldig warten ohne neuen Start
        for _ in range(MAX_START_RETRIES):
            time.sleep(RETRY_WAIT_SEC)
            if is_port_open(host_ip, port, timeout=SOCKET_TIMEOUT):
                try:
                    req = urllib.request.Request(f"{target_host}/api/tags")
                    with urllib.request.urlopen(req, timeout=API_TIMEOUT) as resp:
                        if resp.status == 200:
                            return True
                except Exception:
                    pass
        return False
    # 3. Schritt: Wirklich nicht da -> Erst jetzt einmalig im Hintergrund starten (ohne sichtbares Fenster)
    try:
        if os.name == "nt":
            flags = 0x08000000 | 0x00000200 | 0x00000008  # CREATE_NO_WINDOW Flags
            subprocess.Popen(["ollama", "serve"], creationflags=flags, close_fds=True)
        else:
            subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    except FileNotFoundError:
        print(f"\n[HARDCORE-FEHLER] Der Befehl 'ollama' wurde im System-PATH nicht gefunden. Bitte Ollama installieren.")
        return False
    except Exception as e:
        print(f"\n[HARDCORE-FEHLER] Unerwarteter Systemfehler beim Starten des Ollama-Prozesses: {e}")
        return False
    # 4. Schritt: Warten / Polling bis der neu gestartete Server erreichbar ist
    for _ in range(MAX_START_RETRIES):
        time.sleep(RETRY_WAIT_SEC)
        if is_port_open(host_ip, port, timeout=SOCKET_TIMEOUT):
            try:
                req = urllib.request.Request(f"{target_host}/api/tags")
                with urllib.request.urlopen(req, timeout=API_TIMEOUT) as resp:
                    if resp.status == 200:
                        return True
            except Exception:
                pass
    print(f"\n[HARDCORE-FEHLER] Timeout: Der Ollama-Dienst konnte nach {MAX_START_RETRIES * RETRY_WAIT_SEC} Sekunden nicht erfolgreich gestartet werden.")
    return False

def get_available_local_models() -> dict:
    """
    Ruft alle aktiv bei Ollama registrierten Modelle ab und prüft 
    das lokale Projektverzeichnis direkt über die globalen Variablen.
    """
    print('get_available_local_models')
    local_models_dict = {}
    try:
        client = ollama.Client(host=OLLAMA_HOST)
        response = client.list()
        models_info = response.get('models', []) if isinstance(response, dict) else getattr(response, 'models', [])
        if not models_info:
            print("  Keine Modelle in Ollama registriert gefunden.")
        for m in models_info:
            name = m.get('model') if isinstance(m, dict) else getattr(m, 'model', str(m))
            size_bytes = m.get('size', 0) if isinstance(m, dict) else getattr(m, 'size', 0)
            if name:
                size_gb = size_bytes / (1024 ** 3) if size_bytes else 0.0
                local_models_dict[name] = size_gb
                print(f"  - {name} ({size_gb:.2f} GB)")
    except Exception as e:
        print(f"[Fehler beim Abrufen der Ollama-Modelle von {OLLAMA_HOST}]: {e}")
    models_path = Path(PROJECT_MODELS_DIR)
    if not models_path.exists() and Path("data/models").exists():
        models_path = Path("data/models")
    if models_path.exists() and models_path.is_dir():
        gguf_files = list(models_path.rglob("*.gguf"))
        if gguf_files:
            for file_path in gguf_files:
                size_gb = file_path.stat().st_size / (1024 ** 3)
                print(f"  - {file_path.name} ({size_gb:.2f} GB)")
        else:
            print(f"  Keine .gguf-Modelldateien in '{models_path.resolve()}' gefunden.")
    else:
        print(f"  Verzeichnis '{PROJECT_MODELS_DIR}' wurde nicht gefunden.")
    return local_models_dict

def select_active_models(query: str) -> list:
    """
    Ermittelt den Schwerpunkt der Benutzeranfrage und wählt 
    die passenden Modelle direkt aus den globalen Variablen aus.
    """
    print('select_active_models')
    normalized_query = query.lower()
    is_code = any(kw in normalized_query for kw in ["code", "python", "script", "funktion", "bug", "programm"])
    is_logic = any(kw in normalized_query for kw in ["warum", "analysiere", "berechne", "logik", "vergleiche", "strategie"])
    if is_code:
        return [OLAMA_MODEL_CODE_LOW, OLAMA_MODEL_CODE_HIGH, OLAMA_MODEL_TEXT_LOW]
    elif is_logic:
        return [OLAMA_MODEL_LOGIC_LOW, OLAMA_MODEL_LOGIC_HIGH, OLAMA_MODEL_TEXT_LOW]
    else:
        return [OLAMA_MODEL_TEXT_LOW, OLAMA_MODEL_LOGIC_LOW, OLAMA_MODEL_TEXT_HIGH]

def check_hardware_capacity() -> bool:
    """
    Prüft den System-RAM und den kombinierten VRAM aller verfügbaren GPUs über nvidia-smi
    """
    print('check_hardware_capacity')
    mem = psutil.virtual_memory()
    free_ram_gb = mem.available / (1024 ** 3)
    total_ram_gb = mem.total / (1024 ** 3)
    print(f"[Hardware-Check] System RAM  - Gesamt: {total_ram_gb:.1f} GB | Frei: {free_ram_gb:.1f} GB")
    vram_sufficient = True
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,nounits,noheader'],capture_output=True, text=True, check=True)
        free_vram_list = [int(x) for x in result.stdout.strip().split('\n') if x.strip()]
        device_count = len(free_vram_list)
        free_vram_gb = sum(free_vram_list) / 1024.0
        print(f"[Hardware-Check] GPU VRAM (Multi-GPU via nvidia-smi) - Freier VRAM gesamt: {free_vram_gb:.1f} GB ({device_count} GPUs aktiv)")
        if free_vram_gb < 6.0:
            print("[Hardware-Check] Warnung: VRAM-Kapazität kritisch (< 6 GB frei).")
            vram_sufficient = False
    except Exception as e:
        print(f"[Hardware-Check] nvidia-smi VRAM-Abfrage fehlgeschlagen: {e}")
        # Fallback auf PyTorch, falls nvidia-smi fehlschlägt
        if TORCH_AVAILABLE and torch.cuda.is_available():
            try:
                device_count = torch.cuda.device_count()
                total_free_vram = 0
                for i in range(device_count):
                    free_bytes, _ = torch.cuda.mem_get_info(i)
                    total_free_vram += free_bytes
                free_vram_gb = total_free_vram / (1024 ** 3)
                print(f"[Hardware-Check] GPU VRAM (PyTorch Fallback) - Freier VRAM gesamt: {free_vram_gb:.1f} GB ({device_count} GPUs aktiv)")
                if free_vram_gb < 6.0:
                    vram_sufficient = False
            except Exception as inner_e:
                print(f"[Hardware-Check] Auch PyTorch VRAM-Abfrage fehlgeschlagen: {inner_e}")
        else:
            print("[Hardware-Check] Keine CUDA-GPU via nvidia-smi oder PyTorch erkannt.")

    if free_ram_gb < 4.0 or not vram_sufficient:
        print("[Hardware-Check] Entscheidung: Modus = SEQUENZIELL (Hardware-Schutz aktiv).")
        return False
    else:
        print("[Hardware-Check] Entscheidung: Modus = PARALLEL (Ausreichend Ressourcen).")
        return True

def validate_and_adjust_models(active_models: list, available_models_dict: dict) -> list:
    """
    Prüft, ob die gewünschten Modelle lokal vorhanden sind (unterstützt Teil- und 
    Toleranzmatching bei Namensabweichungen), bereinigt Leerzeichen/Zeilenumbrüche, 
    verhindert Duplikate und gleicht die Modellgrößen mit den Hardware-Ressourcen ab.
    """
    print('validate_and_adjust_models')
    validated_models = []
    if isinstance(available_models_dict, dict):
        available_names_cleaned = {str(m).strip(): size for m, size in available_models_dict.items()}
        available_names = list(available_names_cleaned.keys())
    else:
        available_names_cleaned = {str(m).strip(): 0.0 for m in available_models_dict}
        available_names = list(available_names_cleaned.keys())
        available_models_dict = available_names_cleaned
    #print("\n Starte Validierung der aktiven Modelle ") 
    for model in active_models:
        clean_model = str(model).strip()
        matched_model = None
        if clean_model in available_names:
            matched_model = clean_model
        else:
            base_query = clean_model.split(':')[0].lower()
            for avail in available_names:
                if base_query in avail.lower():
                    matched_model = avail
                    #print(f"[Info] Fuzzy-Match: '{clean_model}' wurde auf lokalem Modell '{matched_model}' abgebildet.")
                    break
        if matched_model:
            if matched_model not in validated_models:
                model_size = available_models_dict.get(matched_model, 0.0)
                if model_size > 70.0:  # Z.B. das 122B Modell
                    print(f"[Warnung] Modell '{matched_model}' ({model_size:.2f} GB) überschreitet die sicheren Hardware-Kapazitäten!")
                else:
                    validated_models.append(matched_model)
        else:
            print(f"[Warnung] Modell '{clean_model}' nicht auf Festplatte gefunden.")
            if available_names:
                fallback = next((m for m in available_names if m not in validated_models), available_names[0])
                print(f"-> Verwende stattdessen verfügbares Fallback-Modell: {fallback}")
                if fallback not in validated_models:
                    validated_models.append(fallback)
    #print(f"Validierte aktive Modelle: {validated_models}\n")
    return validated_models

def clean_query(query: str, 
                language: str = 'german', 
                additional_stopwords: List[str] = None, 
                exclude_stopwords: List[str] = None) -> str:
    """
    Bereinigt die Benutzeranfrage basierend auf einer Standard-Bibliothek (NLTK),
    erlaubt das Hinzufügen eigener Stopwörter und das Ausschließen 
    bestimmter Wörter von der Löschung (Exclude / Whitelist).
    """
    print('clean_query')
    if additional_stopwords is None:
        additional_stopwords = []
    if exclude_stopwords is None:
        exclude_stopwords = []
    stop_words: Set[str] = set(stopwords.words(language))
    stop_words.update(map(str.lower, additional_stopwords))
    for word in exclude_stopwords:
        stop_words.discard(word.lower())
    words = query.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered_words)

async def query_single_model(model_name: str, query: str) -> dict:
    """
    Führt die Anfrage für ein einzelnes Ollama-Modell aus (direkt & sachlich ohne Smalltalk).
    """
    print('query_single_model')
    system_prompt = (
        "Du bist ein präziser Fachassistent. Antworte direkt, sachlich und auf den Punkt. "
        "Vermeide jeglichen Smalltalk, Begrüßungen oder ausschweifende Einleitungen."
    )
    try:
        response = await asyncio.to_thread(
            ollama.chat,
            model=model_name,messages=[{'role': 'system', 'content': system_prompt},{'role': 'user', 'content': query}],
            options={'temperature': 0.1}
        )
        answer = response['message']['content']
        return {"model": model_name,"answer": answer,"status": "success","token_count": len(answer.split())}
    except Exception as e:
        return {"model": model_name,"answer": f"Fehler bei der Ausführung: {str(e)}","status": "error","token_count": 0}

async def execute_model_queries(models: list, query: str) -> list:
    """
    Steuert die Ausführung basierend auf der kombinierten RAM- und VRAM-Prüfung.
    """
    print('execute_model_queries')
    is_hardware_capable = check_hardware_capacity()
    results = []
    if is_hardware_capable:
        tasks = [query_single_model(model, query) for model in models]
        results = await asyncio.gather(*tasks)
    else:
        for model in models:
            print(f"Führe Modell sequenziell aus: {model}...")
            res = await query_single_model(model, query)
            results.append(res)
    return results

def sanitize_model_name(model_name: str) -> str:
    """Bereinigt den Modellnamen für gültige Python-Variablennamen (ersetzt : und - durch _)."""
    return re.sub(r'[^a-zA-Z0-9_]', '_', model_name)

In [37]:
# Pipeline-Ausführung
if not ensure_ollama_running():
    raise RuntimeError("Pipeline-Abbruch: Ollama ist nicht betriebsbereit.")

raw_query = "Wie wird das Wetter heute wohl in unserem Landkreis Klettgau?"
cleaned_query = clean_query(raw_query, language='german', additional_stopwords=["wohl"], exclude_stopwords=["wetter"])

# Wunsch-Modelle anhand der Anfrage ermitteln
initial_active_models = select_active_models(cleaned_query)

# Gegen lokale Festplatte validieren (gegen 404-Fehler)
local_models = get_available_local_models()
ACTIVE_MODELS = validate_and_adjust_models(initial_active_models, local_models)

print(f"Originale Anfrage : {raw_query}")
print(f"Bereinigte Anfrage: {cleaned_query}")
print(f"Validierte aktive Modelle: {ACTIVE_MODELS}\n")

# Start der Abfragen mit Hardware-Überwachung
evaluation_results = await execute_model_queries(ACTIVE_MODELS, cleaned_query)

print("\n[Pipeline] Erstelle modellspezifische Ergebnis-Behälter...")

# Die Antworten in dedizierte, nachvollziehbare globale Behälter abfangen
for idx, res in enumerate(evaluation_results, start=1):
    model_name = res['model']
    clean_model = sanitize_model_name(model_name)
    answer_text = res['answer']
    status_exec = res['status']
    
    # Payload mit kompletter Nachvollziehbarkeit (Modell + Antwort + Ausführungsstatus)
    payload = {
        "model": model_name,
        "answer": answer_text,
        "status": status_exec
    }
    
    # 1. Modellspezifischer globaler Behälter (z.B. result1_gemma3_4b, result2_deepseek_r1_...)
    dynamic_var_name = f"result{idx}_{clean_model}"
    globals()[dynamic_var_name] = payload
    
    # 2. Universal-Alias für die nachfolgende Logik (result1, result2, result3)
    globals()[f"result{idx}"] = payload
    
    print(f"\n[Behälter zugeteilt] Slot: result{idx} | Modell: {model_name}")
    print(f"  -> Globale Variable: `{dynamic_var_name}` erstellt.")
    print(f"  -> Inhalt-Vorschau: {answer_text[:150]}...")

clean_query
select_active_models
get_available_local_models
  - deepseek-r1-8b:latest (15.26 GB)
  - DeepSeek-R1-0528-Qwen3-8B-BF16.gguf (15.26 GB)
  - Qwen3-Coder-Next-UD-Q4_K_XL.gguf (46.20 GB)
  - Qwen3.5-2B-Artemis-Q5_K_M.gguf (1.30 GB)
validate_and_adjust_models
[Warnung] Modell 'gemma3:4b' nicht auf Festplatte gefunden.
-> Verwende stattdessen verfügbares Fallback-Modell: deepseek-r1-8b:latest
[Warnung] Modell 'qwen3.6:27b' nicht auf Festplatte gefunden.
-> Verwende stattdessen verfügbares Fallback-Modell: deepseek-r1-8b:latest
Originale Anfrage : Wie wird das Wetter heute wohl in unserem Landkreis Klettgau?
Bereinigte Anfrage: Wetter heute Landkreis Klettgau?
Validierte aktive Modelle: ['deepseek-r1-8b:latest']

execute_model_queries
check_hardware_capacity
[Hardware-Check] System RAM  - Gesamt: 31.9 GB | Frei: 17.4 GB
[Hardware-Check] GPU VRAM (Multi-GPU via nvidia-smi) - Freier VRAM gesamt: 11.9 GB (2 GPUs aktiv)
[Hardware-Check] Entscheidung: Modus = PARALLEL (Ausreichend Res

In [5]:
# Quality result detektor 
def honesty_and_hallucination_detector(query: str, response_text: str) -> dict:
    """
    Prüft eine Modellantwort auf Ehrlichkeit und potenzielle Halluzinationen,
    insbesondere bei Echtzeit-/Live-Anfragen ohne aktive Tool-Anbindung.
    """
    text_lower = response_text.lower()
    score = 5
    status = "Normal"
    is_realtime_query = any(kw in query.lower() for kw in ["wetter", "heute", "aktuell", "kurs", "preis", "nachrichten"])
    if is_realtime_query:
        honesty_keywords = ["kein zugriff", "keine aktuellen", "dwd", "wetter.com", "extern", "offizielle quellen", "keine echtdaten"]
        hallucination_indicators = ["°c", "sonnig", "regenfällig", "grad", "teilweise bewölkt", "morgen"]
        if any(term in text_lower for term in honesty_keywords):
            score += 4
            status = "Honest / Correct (No Hallucination)"
        elif any(indicator in text_lower for indicator in hallucination_indicators):
            score -= 5
            status = "Critical: Potential Hallucination (Invented Live Data)"
    return {"score": score,"status": status,"detector": "honesty_and_hallucination_detector"}


def semantic_consensus(model_results: dict) -> dict:
    """
    Vergleicht die Antworten der Modelle untereinander auf inhaltliche Konsistenz
    (z. B. ob mehrere Modelle übereinstimmend zugeben, dass Daten fehlen).
    """
    consensus_scores = {}
    responses = list(model_results.values())
    # Einfacher Heuristik-Ansatz: Erkennen von "keine Daten"-Konsens
    no_data_count = sum(1 for r in responses if any(k in r.lower() for k in ["kein zugriff", "keine aktuellen", "leider habe ich"]))
    total_models = len(responses)
    for model_name, resp in model_results.items():
        resp_lower = resp.lower()
        is_no_data = any(k in resp_lower for k in ["kein zugriff", "keine aktuellen", "leider habe ich"])
        bonus = 0
        # Wenn die Mehrheit der Modelle einig ist, dass keine Daten da sind, bekommt das ehrliche Modell einen Konsens-Bonus
        if no_data_count >= (total_models / 2) and is_no_data:
            bonus = 2
        consensus_scores[model_name] = {
            "consensus_bonus": bonus,
            "status": "In Consensus with peers" if (is_no_data == (no_data_count >= (total_models / 2))) else "Outlier"
        }
    return consensus_scores

def llm_judge(query: str, model_results: dict) -> dict:
    """
    Nutzt ein Schiedsrichter-Prinzip (oder eine strukturierte Logik), 
    um die Qualität der Antworten gegeneinander abzuwägen.
    """
    judgements = {}
    print("\n[LLM-Judge] Starte jurierende Bewertung der Modellantworten...")
    for model_name, response_text in model_results.items():
        # Hier kann später optional ein starkes lokales Modell (z.B. Qwen 27B) als Richter aufgerufen werden
        # Vorerst als strukturierte programmatische Evaluierung:
        length_penalty = 0
        if len(response_text.strip()) < 10:
            length_penalty = -3 # Zu kurze/leere Antworten abstrafen
        judgements[model_name] = {
            "judge_penalty": length_penalty,
            "evaluation_note": "Evaluated successfully by LLM-Judge logic"
        }
    return judgements

def query_relevance_evaluator(query: str, response_text: str) -> dict:
    """
    Prüft, ob die Schlüsselbegriffe der bereinigten Anfrage tatsächlich 
    im Fokus der Antwort stehen oder ob das Modell am Thema vorbeiredet.
    Gibt einen normierten Score von 0.0 bis 100.0 % zurück.
    """
    query_terms = set(re.findall(r'\w+', query.lower()))
    text_lower = response_text.lower()
    if not query_terms:
        score = 50.0
    else:
        matched_terms = sum(1 for term in query_terms if term in text_lower)
        score = min(100.0, (matched_terms / len(query_terms)) * 100.0)
        
    status = "High Relevance" if score >= 75.0 else "Partial Relevance" if score >= 40.0 else "Off-Topic / Low Relevance"
    return {"score": round(score, 1),"status": status,"detector": "query_relevance_evaluator"}

def substance_and_utility_evaluator(query: str, response_text: str) -> dict:
    """
    Bewertet die Informationsdichte und den konkreten Nutzwert der Antwort
    (z. B. Nennung verlässlicher Quellen oder verwertbarer Details statt Fülltext).
    Gibt einen normierten Score von 0.0 bis 100.0 % zurück.
    """
    text_lower = response_text.lower()
    words = re.findall(r'\w+', text_lower)
    word_count = len(words)
    utility_keywords = [
        "dwd.de", "wetter.com", "kachelmannwetter", "app", "radar", 
        "warnungen", "quelle", "offiziell", "beispiel", "schritt"
    ]
    utility_hits = sum(1 for kw in utility_keywords if kw in text_lower)
    if word_count < 8:
        score = 15.0  # Einzeiler / kaum Substanz
    else:
        score = min(100.0, 40.0 + (utility_hits * 20.0) + min(20.0, word_count * 0.5))
    status = "High Utility / Actionable" if score >= 75.0 else "Moderate Substance" if score >= 45.0 else "Low Substance / Superficial"
    return {"score": round(score, 1),"status": status,"detector": "substance_and_utility_evaluator"}

def structure_and_completeness_evaluator(response_text: str) -> dict:
    """
    Prüft die sprachliche Vollständigkeit, Sauberkeit des Satzabschlusses
    und strukturelle Integrität der Antwort (erkennt abgeschnittene Outputs).
    Gibt einen normierten Score von 0.0 bis 100.0 % zurück.
    """
    cleaned_text = response_text.strip()
    words = re.findall(r'\w+', cleaned_text)
    word_count = len(words)
    if word_count < 5:
        score = 10.0
        status = "Fragmented / Too Short"
    elif cleaned_text.endswith(('.', '!', '?', '"', ')')):
        score = 95.0
        status = "Complete & Well-Structured"
    else:
        score = 40.0
        status = "Truncated / Incomplete Ending"
    return {"score": round(score, 1),"status": status,"detector": "structure_and_completeness_evaluator"}

def user_expectation_evaluator(raw_query: str, response_text: str) -> dict:
    """
    Prüft, ob die Antwort die tatsächliche Erwartungshaltung und Absicht (Intent)
    des Benutzers aus der Originalanfrage umfassend erfüllt und stützt.
    Gibt einen normierten Score von 0.0 bis 100.0 % zurück.
    """
    q_lower = raw_query.lower()
    r_lower = response_text.lower()
    words = re.findall(r'\w+', r_lower)
    score = 50.0  # Neutraler Ausgangspunkt
    reasons = []
    is_live_expectation = any(kw in q_lower for kw in ["wie wird", "aktuell", "heute", "jetzt", "kurs", "preis"])
    is_howto_expectation = any(kw in q_lower for kw in ["wie kann", "wie mache", "code", "beispiel", "erkläre"])
    if is_live_expectation:
        has_concrete_values = any(u in r_lower for u in ["°c", "grad", "regenwahrscheinlichkeit", "niederschlag", "uv-index"])
        has_direct_solution_path = any(src in r_lower for src in ["dwd.de", "wetter.com", "kachelmannwetter", "wetterdienst"])
        is_evasive = any(ev in r_lower for ev in ["gemäßigten", "klima", "je nach jahreszeit", "kann variieren"]) and not has_direct_solution_path
        if has_concrete_values:
            score += 40.0
            reasons.append("Direkte Daten-Erwartung bedient")
        elif has_direct_solution_path:
            # Ehrliche Absage + konkreter Lösungsweg erfüllt die Erwartung zumindest teilweise (70%)
            score = 70.0
            reasons.append("Ersatz-Lösungsweg statt Live-Daten geliefert")
        elif is_evasive:
            score -= 30.0
            reasons.append("Ausweichend: Allgemeine Hintergrundinfos statt erwarteter Antwort")
        else:
            score -= 20.0
            reasons.append("Erwartungshaltung (Live-Auskunft) nicht erfüllt")
    elif is_howto_expectation:
        if "```" in response_text or "schritt" in r_lower or len(words) > 40:
            score += 35.0
            reasons.append("Umfassende Erklärung/Lösung geliefert")
        else:
            score -= 20.0
            reasons.append("Zu oberflächlich für die gestellte Frage")
    proper_nouns = [w for w in re.findall(r'\b[A-ZÄÖÜ][a-zäöüß]{3,}\b', raw_query) if w.lower() not in ["wie", "was", "wann", "wo", "warum", "unserem", "landkreis"]]
    if proper_nouns:
        matched_entities = sum(1 for ent in proper_nouns if ent.lower() in r_lower)
        if matched_entities == len(proper_nouns):
            score += 10.0
        else:
            score -= 15.0
            reasons.append(f"Kernbezug ({', '.join(proper_nouns)}) fehlt")
    score = max(0.0, min(100.0, score))
    status = "Expectation Fulfilled" if score >= 75.0 else "Partially Fulfilled" if score >= 50.0 else "Expectation Missed (Evasive / Incomplete)"
    return {"score": round(score, 1),"status": status,"notes": "; ".join(reasons) if reasons else "Standard-Abdeckung","detector": "user_expectation_evaluator"}

def truth_and_logic_verifier(raw_query: str,response_text: str,local_knowledge_context: str = "") -> dict:
    """
    8. Dimension (100% Offline, kein LLM-Raten):
    Prüft die faktische Wahrheit, logische Widerspruchsfreiheit (NLI),
    Entitäten-Treue (spaCy) und mathematische Korrektheit (SymPy).
    Gibt einen deterministischen Score von 0.0 bis 100.0 % zurück.
    """
    score = 100.0
    audit_log = []
    #  1. NLI-Logikprüfung (Entailment vs. Contradiction)
    premise = local_knowledge_context.strip() if local_knowledge_context.strip() else raw_query
    scores = nli_model.predict([(premise, response_text)])[0]
    # Reihenfolge bei mDeBERTa-mnli-xnli: [Contradiction, Neutral, Entailment]
    contradiction_prob, neutral_prob, entailment_prob = float(scores[0]), float(scores[1]), float(scores[2])
    if contradiction_prob > 0.50:
        score -= 60.0
        audit_log.append(f"NLI-Widerspruch erkannt ({contradiction_prob:.0%})")
    elif local_knowledge_context and entailment_prob < 0.40:
        score -= 35.0
        audit_log.append(f"Aussage nicht durch lokale Wissensbasis gedeckt (Entailment: {entailment_prob:.0%})")

    #  2. Entitäten- & Zahlen-Halluzinations-Check (spaCy) 
    doc_resp = nlp_de(response_text)
    doc_ref = nlp_de(premise)
    ref_entities = {ent.text.lower() for ent in doc_ref.ents}
    unverified_entities = [ent.text for ent in doc_resp.ents if ent.label_ in ["LOC", "PER", "ORG"] and ent.text.lower() not in ref_entities]
    if not local_knowledge_context and unverified_entities:
        # Bekannte offizielle Quellen erlauben
        allowed = {"dwd", "deutscher wetterdienst", "wetter.com", "kachelmannwetter"}
        suspicious = [e for e in unverified_entities if e.lower() not in allowed]
        if suspicious:
            score -= min(30.0, len(suspicious) * 15.0)
            audit_log.append(f"Unbelegte Entitäten: {', '.join(suspicious)}")

    #  3. Deterministische Rechenprüfung (SymPy)
    math_matches = re.findall(r'(\d+(?:\.\d+)?\s*[\+\-\*\/]\s*\d+(?:\.\d+)?)\s*=\s*(\d+(?:\.\d+)?)', response_text)
    for expr_str, claimed_val_str in math_matches:
        try:
            exact_val = float(sympify(expr_str).evalf())
            claimed_val = float(claimed_val_str)
            if abs(exact_val - claimed_val) > 1e-4:
                score -= 50.0
                audit_log.append(f"Rechenfehler bewiesen: {expr_str} ist {exact_val}, nicht {claimed_val}")
        except Exception:
            pass
    final_score = max(0.0, min(100.0, round(score, 1)))
    status = "Verified Truth" if final_score >= 75.0 else "Unverified / Doubtful" if final_score >= 50.0 else "False / Contradiction"
    return {"score": final_score,"status": status,"notes": "; ".join(audit_log) if audit_log else "Deterministisch konsistent","detector": "truth_and_logic_verifier"}

In [6]:
# Plot score
def plot_polar_evaluation(scored_models: dict, save_path: str = None):
    """
    Autonomer, selbstjustierender N-achsiger Polar-Plot (0% bis 100%).
    Erkennt alle Bewertungs-Dimensionen dynamisch aus `scored_models`,
    justiert Winkel, Achsen-Labels, Farben und Figur-Layout vollautomatisch.
    """
    if not scored_models:
        print("[Visualisierung] Warnung: Keine Daten in `scored_models` vorhanden.")
        return

    # 1. Dimensionen dynamisch aus den Daten ermitteln
    first_model = next(iter(scored_models.values()))
    categories = list(first_model.get("dimensions", {}).keys())
    num_vars = len(categories)

    if num_vars < 3:
        print(f"[Visualisierung] Mindestens 3 Dimensionen für Polar-Plot benötigt (aktuell: {num_vars}).")
        return

    # 2. Winkel dynamisch für N Achsen berechnen und Polygon schließen
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    # 3. Selbstjustierende Figur-Größe
    num_models = len(scored_models)
    base_size = max(9, 8 + (num_vars * 0.25))
    fig_width = base_size + 9
    fig_height = base_size + (num_models * 0.05) - 5

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), subplot_kw=dict(polar=True), facecolor='white')
    ax.set_facecolor('white')

    # 4. Dynamische Farbpalette
    base_colors = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e', '#8c564b', '#e377c2']
    if num_models > len(base_colors):
        cmap = cm.get_cmap('tab10', num_models)
        colors = [cmap(i) for i in range(num_models)]
    else:
        colors = base_colors[:num_models]

    # 5. Modelle dynamisch einzeichnen
    for idx, (model_name, model_data) in enumerate(scored_models.items()):
        cont = model_data.get("container", f"Slot {idx+1}")
        overall = model_data.get("overall_pct", 0.0)
        dims = model_data.get("dimensions", {})
        values = [float(dims.get(cat, 0.0)) for cat in categories]
        values += values[:1]
        color = colors[idx]
        label = f"{cont}: {model_name} (Ø {overall}%)"
        ax.plot(angles, values, color=color, linewidth=2.5, marker='o', markersize=5, label=label)
        ax.fill(angles, values, color=color, alpha=0.14)

    # 6. Ausrichtung
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11, fontweight='bold')

    # 7. Selbstjustierende Textausrichtung der Achsen-Labels (verhindert Überlappung mit dem Kreis)
    ax.tick_params(axis='x', pad=14)
    for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
        deg = np.degrees(angle) % 360
        if deg == 0 or deg == 180:
            label.set_horizontalalignment('center')
        elif 0 < deg < 180:
            label.set_horizontalalignment('left')
        else:
            label.set_horizontalalignment('right')

    # Radiale Skalierung (0% bis 100%)
    ax.set_rlabel_position(360 / (num_vars * 2))  # Platziert Prozentzahlen sauber zwischen zwei Achsen
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], color="dimgray", fontsize=9)
    ax.set_ylim(0, 100)

    # Kritische 50%-Schwelle als gestrichelter Ring
    circle_angles = np.linspace(0, 2 * np.pi, 120)
    ax.plot(circle_angles, [50] * 120, color='gray', linestyle='--', linewidth=1.2, alpha=0.75, label='Kritische Schwelle (50%)')

    plt.title(f"Multi-LLM Qualitäts- & Diagnose-Profil ({num_vars} Dimensionen)", size=14, fontweight='bold', pad=32)

    # Legende sauber außerhalb rechts platzieren
    plt.legend(loc='center left',bbox_to_anchor=(1.22, 0.5),frameon=True,facecolor='white',edgecolor='lightgray',fontsize=10)
    plt.tight_layout()

    plt.show()

In [11]:
# 8-Dimensionale Qualitäts-, Wahrheits- & Konsens-Evaluation & Tabellen-Ausgabe
print("\n[Evaluation Pipeline] Sammle Result-Behälter ein und starte 8-Achsen-Qualitätsprüfung...")

# 1. Die aktiven Result-Behälter dynamisch & robust einsammeln
active_results = {}
model_responses_dict = {}

# Prüfe flexibel von result1 bis result5 (oder falls als Liste/Dict im Scope)
for i in range(1, 6):
    res_key = f"result{i}"
    if res_key in globals() and isinstance(globals()[res_key], dict):
        res_data = globals()[res_key]
        if "model" in res_data and "answer" in res_data:
            model_name = res_data["model"]
            answer_text = res_data["answer"]
            active_results[res_key] = res_data
            model_responses_dict[model_name] = answer_text

print(f"[Evaluation Pipeline] {len(active_results)} aktive Result-Behälter erfolgreich eingesammelt.")

# Optionaler lokaler Wissenskontext (falls vorhanden, sonst leerer String)
verified_context = globals().get("local_knowledge_context", "")

# 2. Ausführung aller 8 Qualitäts-Defs
print("\n A) Ehrlichkeits- & Halluzinations-Detektor ")
hallucination_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    eval_res = honesty_and_hallucination_detector(cleaned_query, data["answer"])
    hallucination_evals[model_name] = eval_res
    print(f"[{res_key}] {model_name} -> Raw-Score: {eval_res['score']} | {eval_res['status']}")

print("\n B) Semantischer Konsens ")
consensus_evals = semantic_consensus(model_responses_dict)
for model_name, c_data in consensus_evals.items():
    print(f"Modell: {model_name} -> Bonus: {c_data['consensus_bonus']} | {c_data['status']}")

print("\n C) LLM-Judge (Logik & Schiedsrichter) ")
judge_evals = llm_judge(cleaned_query, model_responses_dict)
for model_name, j_data in judge_evals.items():
    print(f"Modell: {model_name} -> Penalty: {j_data['judge_penalty']} | {j_data['evaluation_note']}")

print("\n D) Anfrage-Relevanz ")
relevance_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    rel_res = query_relevance_evaluator(cleaned_query, data["answer"])
    relevance_evals[model_name] = rel_res
    print(f"[{res_key}] {model_name} -> {rel_res['score']}% | {rel_res['status']}")

print("\n E) Substanz & Nutzwert ")
substance_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    sub_res = substance_and_utility_evaluator(cleaned_query, data["answer"])
    substance_evals[model_name] = sub_res
    print(f"[{res_key}] {model_name} -> {sub_res['score']}% | {sub_res['status']}")

print("\n F) Struktur & Vollständigkeit ")
structure_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    str_res = structure_and_completeness_evaluator(data["answer"])
    structure_evals[model_name] = str_res
    print(f"[{res_key}] {model_name} -> {str_res['score']}% | {str_res['status']}")

print("\n G) Benutzer-Erwartung & Intent-Stützung (7. Dimension) ")
expectation_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    exp_res = user_expectation_evaluator(raw_query, data["answer"])
    expectation_evals[model_name] = exp_res
    print(f"[{res_key}] {model_name} -> {exp_res['score']}% | {exp_res['status']} ({exp_res['notes']})")

print("\n H) Deterministische Wahrheits- & Logikprüfung (8. Dimension - Offline NLI/spaCy/SymPy) ")
truth_evals = {}
for res_key, data in active_results.items():
    model_name = data["model"]
    truth_res = truth_and_logic_verifier(raw_query, data["answer"], local_knowledge_context=verified_context)
    truth_evals[model_name] = truth_res
    print(f"[{res_key}] {model_name} -> {truth_res['score']}% | {truth_res['status']} ({truth_res['notes']})")


# 3. Normierung auf 0 - 100 % & Zusammenführung für den 8-Achsen-Polar-Plot
print("\n[Evaluation Pipeline] Normiere alle 8 Dimensionen auf 0% - 100%...")
final_scored_models = {}

for res_key, data in active_results.items():
    model_name = data["model"]
    
    # Schutz vor KeyError falls ein Dictionary-Key fehlen sollte
    raw_honesty = hallucination_evals.get(model_name, {}).get("score", 0)
    honesty_pct = max(0.0, min(100.0, (raw_honesty / 9.0) * 100.0))
    
    c_bonus = consensus_evals.get(model_name, {}).get("consensus_bonus", 0)
    consensus_pct = 100.0 if c_bonus > 0 else 30.0
    
    j_penalty = judge_evals.get(model_name, {}).get("judge_penalty", 0)
    judge_pct = max(0.0, min(100.0, 100.0 + (j_penalty * 25.0)))
    
    relevance_pct = relevance_evals.get(model_name, {}).get("score", 0.0)
    substance_pct = substance_evals.get(model_name, {}).get("score", 0.0)
    structure_pct = structure_evals.get(model_name, {}).get("score", 0.0)
    expectation_pct = expectation_evals.get(model_name, {}).get("score", 0.0)
    truth_pct = truth_evals.get(model_name, {}).get("score", 0.0)
    
    overall_pct = round(
        (truth_pct * 0.25) +
        (honesty_pct * 0.20) +
        (expectation_pct * 0.15) +
        (consensus_pct * 0.10) +
        (relevance_pct * 0.10) +
        (substance_pct * 0.10) +
        (judge_pct * 0.05) +
        (structure_pct * 0.05),
        1
    )
    
    final_scored_models[model_name] = {
        "container": res_key,
        "overall_pct": overall_pct,
        "truth_notes": truth_evals.get(model_name, {}).get("notes", ""),
        "dimensions": {
            "Ehrlichkeit": round(honesty_pct, 1),
            "Konsens": round(consensus_pct, 1),
            "LLM-Judge": round(judge_pct, 1),
            "Relevanz": round(relevance_pct, 1),
            "Substanz": round(substance_pct, 1),
            "Struktur": round(structure_pct, 1),
            "Erwartung": round(expectation_pct, 1),
            "Wahrheit": round(truth_pct, 1)
        }
    }
    print(f"  -> [{res_key}] {model_name}: Gesamtqualität = {overall_pct}%")

# 4. Tabellarische Übersicht der Ergebnisse ausgeben (mittels pandas)
def print_evaluation_table(scored_models: dict):
    if not scored_models:
        print("[Tabelle] Keine Daten vorhanden.")
        return
    
    rows = []
    for m_name, d in scored_models.items():
        row = {
            "Modell": m_name,
            "Container": d["container"],
            "Gesamt (%)": d["overall_pct"]
        }
        row.update(d["dimensions"])
        rows.append(row)
        
    df = pd.DataFrame(rows)
    df = df.sort_values(by="Gesamt (%)", ascending=False)
    
    print("\n" + "="*95)
    print(" 📊 FINALE 8-ACHSEN-BEWERTUNG DER MODELLANTWORTEN (ÜBERSICHT) ")
    print("="*95)
    print(df.to_string(index=False))
    print("="*95 + "\n")

# Tabelle direkt in der Konsole anzeigen
print_evaluation_table(final_scored_models)

# 5. Autonomen Polar-Plot aufrufen
if 'plot_polar_evaluation' in globals():
    try:
        plot_polar_evaluation(final_scored_models)
    except Exception as e:
        print(f"[Plot-Fehler] Konnte Polar-Plot nicht rendern: {e}")
else:
    print("[Hinweis] `plot_polar_evaluation` ist in diesem Scope nicht definiert.")


[Evaluation Pipeline] Sammle Result-Behälter ein und starte 8-Achsen-Qualitätsprüfung...
[Evaluation Pipeline] 0 aktive Result-Behälter erfolgreich eingesammelt.

 A) Ehrlichkeits- & Halluzinations-Detektor 

 B) Semantischer Konsens 

 C) LLM-Judge (Logik & Schiedsrichter) 

[LLM-Judge] Starte jurierende Bewertung der Modellantworten...

 D) Anfrage-Relevanz 

 E) Substanz & Nutzwert 

 F) Struktur & Vollständigkeit 

 G) Benutzer-Erwartung & Intent-Stützung (7. Dimension) 

 H) Deterministische Wahrheits- & Logikprüfung (8. Dimension - Offline NLI/spaCy/SymPy) 

[Evaluation Pipeline] Normiere alle 8 Dimensionen auf 0% - 100%...
[Tabelle] Keine Daten vorhanden.
[Visualisierung] Warnung: Keine Daten in `scored_models` vorhanden.


In [12]:
# fusion quality models
def extractive_synthesis(final_scored_models: dict, active_results: dict, similarity_threshold: float = 0.70) -> dict:
    """
    Führt eine rein deterministische, extraktive Synthese durch:
    - Nimmt das Modell mit dem höchsten Gesamtscore als Basis-Anker.
    - Filtert alle Modelle unter 50% aus.
    - Extrahiert neue, nicht-redundante Sätze/Fakten aus den Partner-Modellen 
      und reichert den Basis-Text strukturiert an (ohne LLM-Halluzination).
    """
    if not final_scored_models:
        print("[Synthese] Fehler: Keine scored_models vorhanden.")
        return {"synthesized_answer": ""}

    # 1. Bestes Modell ermitteln (höchster overall_pct)
    sorted_models = sorted(final_scored_models.items(), key=lambda x: x[1]["overall_pct"], reverse=True)
    best_model_name, best_data = sorted_models[0]
    best_container = best_data["container"]
    base_answer = active_results[best_container]["answer"]

    #print(f"\n[Synthese-Engine] Basis-Anker gewählt: {best_model_name} ({best_data['overall_pct']}%) aus Container `{best_container}`")

    # 2. Partner-Modelle filtern, die die 50%-Schwelle überschreiten
    qualified_partners = [(m_name, m_data) for m_name, m_data in sorted_models[1:] if m_data["overall_pct"] >= 50.0]
    added_details = []
    def sentence_similarity(a: str, b: str) -> float:
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()
    base_sentences = [s.strip() for s in base_answer.replace('!', '.').replace('?', '.').split('.') if len(s.strip()) > 10]

    # 3. Sätze der Partner-Modelle prüfen und bei Neuheitswert hinzufügen
    for m_name, m_data in qualified_partners:
        m_container = m_data["container"]
        m_answer = active_results[m_container]["answer"]
        m_sentences = [s.strip() for s in m_answer.replace('!', '.').replace('?', '.').split('.') if len(s.strip()) > 10]
        for sentence in m_sentences:
            is_new = all(sentence_similarity(sentence, base_s) < similarity_threshold for base_s in base_sentences)
            if is_new:
                added_details.append((m_name, sentence))
                base_sentences.append(sentence)

    # 4. Finalen Synthese-Text deterministisch zusammenbauen
    synthesized_text = base_answer.rstrip()
    if added_details:
        synthesized_text += "\n\n Ergänzende Details\n" #aus qualifizierten Partner-Modellen \n"
        for m_name, detail in added_details:
            synthesized_text += f"{detail}.\n" # • [{m_name}]:"
    return {"best_model": best_model_name,"base_container": best_container,"qualified_partners": [m[0] for m in qualified_partners],"synthesized_answer": synthesized_text,"added_details_count": len(added_details)}

#  Ausführung der Synthese 
synthesis_dict = extractive_synthesis(final_scored_models, active_results)
synthesized_text_input = synthesis_dict.get("synthesized_answer", "")
print(synthesized_text_input)

[Synthese] Fehler: Keine scored_models vorhanden.



In [ ]:
# Result fusion
def clean_and_fuse_text(synthesized_text_input: str, _cfg: dict) -> str:
    """
    Vereinfachter, extrem robuster Fusions-Prompt für kleine 2B-Modelle.
    """
    if not synthesized_text_input or not synthesized_text_input.strip():
        return ""
    model_name = _cfg.setdefault("OLAMA_MODEL_FUSION", "qwen3.5-2b-artemis-q5-k-m")
    print(f"[Fusions-Agent] Starte Fusion mit Modell: {model_name}...")
    prompt = f"""Schreibe aus den folgenden Informationen einen einzigen, fließenden und sauberen Absatz für den Landkreis Klettgau. 
Lass alle Hinweise auf fehlende Echtzeitdaten oder externe Websites komplett weg.

Quelltext:
{synthesized_text_input}

Fließtext:"""
    system_prompt = "Du bist ein präziser Redakteur. Formuliere die Fakten zu einem einzigen sauberen Text um."
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": 0.2,  # Etwas Kreativität für den Sprachfluss
                "num_ctx": 4096
            }
        )
        result_text = response.get('message', {}).get('content', '').strip()
        if not result_text:
            print("[Fusions-Agent Warnung] Wieder leer – liefere bereinigten Text als Fallback.")
            return synthesized_text_input
        return result_text
    except Exception as e:
        print(f"[Fusions-Agent Fehler]: {e}")
        return synthesized_text_input

In [ ]:
# Ausführung in deiner Pipeline
final_clean_output = clean_and_fuse_text(synthesized_text_input, _cfg)

# 4. Schritt: Das finale Ergebnis ausgeben
print(final_clean_output)